In [ ]:
%restart_python

In [ ]:
# Cell 1: Load raw Spark table
df = spark.sql("SELECT * FROM melodatabricks616.default.yara_dump_table")
print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")

In [ ]:
# Cell 2: Clone repo
import os

REPO_URL = "https://github.com/attabeezy/seqcredit-model.git"
BRANCH   = "real-data"
REPO_DIR = "/tmp/seqcredit-model"

os.system(f"rm -rf {REPO_DIR}")
os.system(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
%pip install -r /tmp/seqcredit-model/requirements.txt
%pip install -e /tmp/seqcredit-model

In [ ]:
%restart_python

In [ ]:
# Cell 5: Reload raw table (post-restart)
df = spark.sql("SELECT * FROM melodatabricks616.default.yara_dump_table")
print(f"Rows: {df.count():,}  |  Columns: {len(df.columns)}")

In [ ]:
# Cell 6: Build LSTM sequences
import os, importlib

os.system("git -C /tmp/seqcredit-model pull")

import seqcredit_model.real_data_pipeline as rdp
importlib.reload(rdp)

seq_path = rdp.build_sequences_spark(df, min_followup_days=60, max_seq_len=100)
print(f"\nSequences saved to: {seq_path}")

In [ ]:
# Cell 7: Build pipeline (features + labels)
import seqcredit_model.real_data_pipeline as rdp

features_df, labels_df = rdp.build_pipeline(df)
display(labels_df["credit_risk_label"].value_counts().sort_index().reset_index())

In [ ]:
# Cell 8: Full CV benchmark (all models, both targets)
import os, importlib

os.system("git -C /tmp/seqcredit-model pull")

import seqcredit_model.run_cv_benchmark as bench
importlib.reload(bench)

bench.main()

In [ ]:
# Cell 9: Display CV results (y_default focus)
import pandas as pd
from seqcredit_model.run_cv_benchmark import get_runtime_data_dir

runtime_dir = get_runtime_data_dir()

cv_default = pd.read_csv(runtime_dir / "cv_results_y_default.csv")
cv_bad     = pd.read_csv(runtime_dir / "cv_results_y_bad.csv")

summary = (
    cv_default
    .groupby("model")[["auc_roc", "auc_pr", "f1", "brier", "ece"]]
    .mean()
    .round(4)
    .sort_values("auc_roc", ascending=False)
)

print("=== CV Results — y_default (mean across 5 folds) ===")
print(summary.to_string())

print("\n=== CV Results — y_bad (mean across 5 folds) ===")
print(
    cv_bad
    .groupby("model")[["auc_roc", "auc_pr", "f1", "brier", "ece"]]
    .mean().round(4).sort_values("auc_roc", ascending=False).to_string()
)

In [ ]:
# Cell 10: Best model selection (rank-sum) on y_default
#
# Each model is ranked on three metrics computed from the y_default CV means:
#   AUC-ROC  rank: ascending=False  (rank 1 = highest AUC)
#   ECE      rank: ascending=True   (rank 1 = lowest ECE)
#   Brier    rank: ascending=True   (rank 1 = lowest Brier)
#
# Winner = lowest total rank.
# If the winner is a sequence model (LSTM/GRU/Transformer), the ablation
# falls back to the best-ranking static model — feature-group ablation
# requires named static features and does not generalise to padded sequences.

STATIC_MODELS = {"LogisticRegression", "XGBoost", "RandomForest", "LightGBM"}

rank_df = summary[["auc_roc", "ece", "brier"]].copy()
rank_df["rank_auc"]   = rank_df["auc_roc"].rank(ascending=False).astype(int)
rank_df["rank_ece"]   = rank_df["ece"].rank(ascending=True).astype(int)
rank_df["rank_brier"] = rank_df["brier"].rank(ascending=True).astype(int)
rank_df["rank_total"] = rank_df[["rank_auc", "rank_ece", "rank_brier"]].sum(axis=1)
rank_df = rank_df.sort_values("rank_total")

print("=== Rank-Sum Scores — y_default (lower total = better) ===")
print(rank_df.to_string())

best_overall = rank_df.index[0]

if best_overall in STATIC_MODELS:
    ablation_model = best_overall
    print(f"\nWinner: {best_overall}  →  ablation will use {ablation_model}")
else:
    static_rank = rank_df[rank_df.index.isin(STATIC_MODELS)]
    ablation_model = static_rank.index[0]
    print(f"\nWinner: {best_overall} (sequence model — feature-group ablation not applicable)")
    print(f"Ablation fallback: {ablation_model} (best static model by rank-sum)")

In [ ]:
# Cell 11: Feature-group ablation on winning (or fallback) model
#
# Runs 5-fold CV (y_default) on the ablation model under three conditions:
#   [1/3] ALL_FEATURES  — baseline
#   [2/3] DROP_<group>  — one feature group removed at a time
#   [3/3] ONLY_<group>  — only that group's features kept
#
# Output: /tmp/seqcredit_model/ablation_notebook_c.csv

import time
import numpy as np
import pandas as pd

from seqcredit_model.credit_model import (
    CreditRiskDataLoader,
    LogisticRegressionModel,
    XGBoostModel,
    RandomForestModel,
    LightGBMModel,
    set_random_seeds,
)
from seqcredit_model.run_cv_benchmark import run_static_model_cv, get_runtime_data_dir
from seqcredit_model.run_ablation_study import FEATURE_GROUPS

set_random_seeds(42)

runtime_dir = get_runtime_data_dir()

loader = CreditRiskDataLoader(
    features_path=str(runtime_dir / "user_features.csv"),
    summaries_path=str(runtime_dir / "user_labels.csv"),
)
static_data = loader.prepare_static_splits()
X_df  = static_data["X_train"]   # DataFrame — column names preserved
y_abl = static_data["y_train"]

# Same params used in run_cv_benchmark
MODEL_REGISTRY = {
    "LogisticRegression": (
        LogisticRegressionModel,
        {"class_weight": "balanced", "C": 1.0, "random_state": 42},
    ),
    "XGBoost": (
        XGBoostModel,
        {"scale_pos_weight": loader.get_scale_pos_weight(), "random_state": 42},
    ),
    "RandomForest": (
        RandomForestModel,
        {"class_weight": "balanced", "random_state": 42},
    ),
    "LightGBM": (
        LightGBMModel,
        {"class_weight": "balanced", "random_state": 42},
    ),
}

model_class, model_params = MODEL_REGISTRY[ablation_model]
print(f"Ablation model : {ablation_model}")
print(f"Training users : {len(y_abl):,}")
print(f"Positive rate  : {y_abl.mean():.4f}")

# Warn on missing feature group columns
all_cols = list(X_df.columns)
for grp, cols in FEATURE_GROUPS.items():
    missing = [c for c in cols if c not in all_cols]
    if missing:
        print(f"  WARNING: {grp} — {len(missing)} columns not found: {missing}")

# Ablation runner
def run_condition(name, X_numpy, y):
    t0 = time.time()
    _, summary, _ = run_static_model_cv(model_class, model_params, X_numpy, y, n_splits=5)
    elapsed = time.time() - t0
    return {
        "condition":    name,
        "n_features":   X_numpy.shape[1],
        "auc_roc":      summary["auc_roc"],
        "auc_roc_std":  summary.get("auc_roc_std",  float("nan")),
        "auc_pr":       summary["auc_pr"],
        "auc_pr_std":   summary.get("auc_pr_std",   float("nan")),
        "f1":           summary["f1"],
        "brier":        summary["brier"],
        "ece":          summary["ece"],
        "elapsed_s":    round(elapsed, 1),
    }

X_np    = X_df.values
records = []

# [1/3] Baseline
print("\n[1/3] Baseline (all features)...")
records.append(run_condition("ALL_FEATURES", X_np, y_abl))
baseline_auc = records[-1]["auc_roc"]
print(f"  AUC-ROC = {baseline_auc:.4f}")

# [2/3] Drop-one-group ablations
print("\n[2/3] Drop-one-group ablations...")
for grp, cols in FEATURE_GROUPS.items():
    drop_idx = [i for i, c in enumerate(all_cols) if c in cols]
    if not drop_idx:
        print(f"  {grp}: no matching columns — skipped")
        continue
    keep_idx = [i for i in range(len(all_cols)) if i not in drop_idx]
    rec      = run_condition(f"DROP_{grp}", X_np[:, keep_idx], y_abl)
    delta    = rec["auc_roc"] - baseline_auc
    records.append(rec)
    print(f"  DROP_{grp}: AUC-ROC = {rec['auc_roc']:.4f}  (\u0394 = {delta:+.4f})")

# [3/3] Single-group-only ablations
print("\n[3/3] Single-group-only ablations...")
for grp, cols in FEATURE_GROUPS.items():
    keep_idx = [i for i, c in enumerate(all_cols) if c in cols]
    if not keep_idx:
        print(f"  {grp}: no matching columns — skipped")
        continue
    rec = run_condition(f"ONLY_{grp}", X_np[:, keep_idx], y_abl)
    records.append(rec)
    print(f"  ONLY_{grp}: AUC-ROC = {rec['auc_roc']:.4f}")

# Save
abl_df   = pd.DataFrame(records)
out_path = runtime_dir / "ablation_notebook_c.csv"
abl_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

In [ ]:
# Cell 12: Display ablation results
import matplotlib.pyplot as plt

drop_rows = abl_df[abl_df["condition"].str.startswith("DROP_")].copy()
drop_rows["group"] = drop_rows["condition"].str.replace("DROP_", "", regex=False)
drop_rows["delta"] = drop_rows["auc_roc"] - baseline_auc
drop_rows = drop_rows.sort_values("delta")   # most negative = most important

print(f"=== Ablation Results — {ablation_model} on y_default ===")
print(
    drop_rows[["group", "n_features", "auc_roc", "delta", "brier", "ece"]]
    .to_string(index=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#d62728" if d < 0 else "#2ca02c" for d in drop_rows["delta"]]
ax.barh(drop_rows["group"], drop_rows["delta"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("\u0394AUC-ROC vs ALL_FEATURES  (negative = group is important)")
ax.set_title(f"Feature Group Ablation — {ablation_model} (y_default)")
plt.tight_layout()
plt.show()